# 原 Stage1 0bp SOXL 相对选择日损失集中度

冻结完整资金账户物理分量，保留全部 944 日、16 季度；只诊断异常日集中度。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_original_0bp_selection_tail_audit_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_requests','model_fits','native_accounts','bootstraps'))
a=root/src['stage1_card'];b=root/src['component_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['stage1_summary_sha256'] and sha(b/'summary.json')==src['component_summary_sha256']
assert sa['recorder_id']==src['stage1_recorder_id'] and sb['recorder_id']==src['component_recorder_id']
assert sa['nas_evidence']['sha256']==src['stage1_NAS_archive_sha256'] and sb['nas_evidence']['sha256']==src['component_NAS_archive_sha256']
assert sha(a/'qlib_recorder'/src['stage1_experiment_id']/src['stage1_recorder_id']/'artifacts/inputs/frame')==src['stage1_frame_sha256']
assert sha(b/'qlib_recorder'/src['component_experiment_id']/src['component_recorder_id']/'artifacts/analysis/daily')==src['component_daily_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['calendar_provider_uri']),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):frame=R.get_recorder(recorder_id=src['stage1_recorder_id'],experiment_id=src['stage1_experiment_id']).load_object('inputs/frame')
with R.uri_context(str(b/'qlib_recorder')):physical=R.get_recorder(recorder_id=src['component_recorder_id'],experiment_id=src['component_experiment_id']).load_object('analysis/daily')
assert len(frame)==len(physical)==944
for c in ('signal_date','trade_date','exit_date'):assert pd.DatetimeIndex(pd.to_datetime(frame[c])).equals(pd.DatetimeIndex(pd.to_datetime(physical[c])))
calendar=pd.DatetimeIndex(D.calendar(start_time=str(frame.signal_date.iloc[0]),end_time=str(frame.signal_date.iloc[-1]),freq='day'))
assert len(calendar)==944 and calendar.equals(pd.DatetimeIndex(pd.to_datetime(frame.signal_date)))
d=pd.DataFrame({'signal_date':pd.to_datetime(frame.signal_date),'trade_date':pd.to_datetime(frame.trade_date),'exit_date':pd.to_datetime(frame.exit_date),'SOXL_selection_log':physical.SOXL_selection_log.astype(float),'cash_opportunity_log':physical.cash_opportunity_log.astype(float),'post_open_soxl_weight':physical.post_open_soxl_weight.astype(float),'saved_price_plane_soxl_minus_soxx':frame.soxl_return.astype(float)-frame.soxx_return.astype(float)})
assert d.notna().all().all() and abs(d.SOXL_selection_log.sum()-0.22593145822639604)<1e-10
static=pd.concat({'label':d[['SOXL_selection_log','cash_opportunity_log','saved_price_plane_soxl_minus_soxx']],'meta':d[['post_open_soxl_weight']]},axis=1);static.index=pd.MultiIndex.from_arrays([d.signal_date,['SOXL_SOXX_CASH']*len(d)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'diagnostic':(str(d.signal_date.iloc[0].date()),str(d.signal_date.iloc[-1].date()))});assert len(dataset.prepare('diagnostic',col_set=['label','meta']))==944
n=int(plan['fixed_statistics']['top_negative_count'])
def describe(g):
 negative=g.loc[g.SOXL_selection_log.lt(0)].sort_values('SOXL_selection_log');denom=float(-negative.SOXL_selection_log.sum());top=negative.head(n);fraction=float(-top.SOXL_selection_log.sum()/denom) if denom>0 else None
 return {'sessions':len(g),'selection_log_sum':float(g.SOXL_selection_log.sum()),'negative_days':len(negative),'positive_days':int(g.SOXL_selection_log.gt(0).sum()),'zero_days':int(g.SOXL_selection_log.eq(0).sum()),'negative_sum':float(negative.SOXL_selection_log.sum()),'positive_sum':float(g.loc[g.SOXL_selection_log.gt(0),'SOXL_selection_log'].sum()),'top10_negative_share_of_all_negative':fraction,'top10_negative_sum':float(top.SOXL_selection_log.sum()),'mean_post_open_soxl_weight':float(g.post_open_soxl_weight.mean()),'top10_negative_dates':[str(x.date()) for x in top.trade_date]}
allstats=describe(d);quarters={str(q):describe(g) for q,g in d.groupby(d.trade_date.dt.to_period('Q'),sort=True)};assert len(quarters)==16
assert abs(quarters['2024Q3']['selection_log_sum']-0.012208238445748431)<1e-10 and abs(quarters['2026Q2']['selection_log_sum']-0.289452743369704)<1e-10 and abs(quarters['2026Q3']['selection_log_sum']+0.14390082802460138)<1e-10
q3=d.loc[d.trade_date.dt.to_period('Q').eq(pd.Period('2026Q3'))];top_q3=q3.nsmallest(n,'SOXL_selection_log');top_all=d.nsmallest(n,'SOXL_selection_log')
report={'decision':'descriptive_original_Stage1_SOXL_selection_daily_concentration_only','all':allstats,'quarters':quarters,'Q3_2026_top10_negative_rows':top_q3.to_dict(orient='records'),'full_top10_negative_rows':top_all.to_dict(orient='records'),'q3_outlier_price_audit_next':quarters['2026Q3']['top10_negative_share_of_all_negative']>=0.70,'model_fits':0,'native_accounts':0,'bootstraps':0,'causal_policy_inference':False}
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='analysis',daily=d,report=report);rec.log_metrics(calendar_days=944,quarter_count=16,q3_negative_days=quarters['2026Q3']['negative_days'],q3_top10_negative_share=quarters['2026Q3']['top10_negative_share_of_all_negative'],fits=0,accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Same physical-account source validated earlier; this is concentration attribution, not independent price validation.','Top negative days are selected for source-quality triage only, never as a trading rule.','2023-2026 outcome history is reused development feedback.']}
(card/'result.json').write_text(json.dumps(report,indent=2,default=str)+'\n');(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='selection-tail-restore-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/daily') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print({'Q3':quarters['2026Q3'],'all':allstats})


2026/10/23 07:06:27 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[13437:MainThread](2026-10-23 07:06:28,894) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13437:MainThread](2026-10-23 07:06:28,896) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13437:MainThread](2026-10-23 07:06:28,897) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[13437:MainThread](2026-10-23 07:06:28,997) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_original_0bp_selection_tail_audit_20261023.


[13437:MainThread](2026-10-23 07:06:28,999) INFO - qlib.workflow - [exp.py:258] - Experiment 687910100685888074 starts running ...


[13437:MainThread](2026-10-23 07:06:29,140) INFO - qlib.workflow - [recorder.py:345] - Recorder 8ea62d29d5f748d6b1673742e91c145c starts running under Experiment 687910100685888074 ...


[13437:MainThread](2026-10-23 07:06:30,193) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'Q3': {'sessions': 64, 'selection_log_sum': -0.14390082802460138, 'negative_days': 8, 'positive_days': 9, 'zero_days': 47, 'negative_sum': -0.14892133342817773, 'positive_sum': 0.005020505403576371, 'top10_negative_share_of_all_negative': 1.0, 'top10_negative_sum': -0.14892133342817773, 'mean_post_open_soxl_weight': 0.02951763407327821, 'top10_negative_dates': ['2026-07-27', '2026-07-06', '2026-07-01', '2026-07-24', '2026-07-02', '2026-07-23', '2026-09-11', '2026-09-25']}, 'all': {'sessions': 944, 'selection_log_sum': 0.22593145822639604, 'negative_days': 336, 'positive_days': 383, 'zero_days': 225, 'negative_sum': -2.178371275081709, 'positive_sum': 2.404302733308105, 'top10_negative_share_of_all_negative': 0.13744670955308497, 'top10_negative_sum': -0.29940996394493896, 'mean_post_open_soxl_weight': 0.1507108992374473, 'top10_negative_dates': ['2026-07-27', '2026-07-06', '2024-03-08', '2025-11-06', '2026-05-14', '2026-06-22', '2025-07-31', '2025-02-27', '2024-06-20', '2023-03-09']}}

## 中文结论

# 原 Stage1 SOXL 选择损失集中度 v1：解释门失效

原生 Qlib 的 944 日物理选择损益与季度闭合仍有效，但 v1 附带的 `frame.soxl_return−frame.soxx_return` 是决策前已知的过去收益，不能解释同一持有期损失。2026Q3 只有 8 个负选择日，固定 top10 占比必然为 100%，不能作为异常日门。保留 v1 Recorder/NAS 和失败证据；v2 只修正来源时间语义、使用 top2/top5 与有效损失日数描述，不拟合或重跑账户。
